In [1]:
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np
import torch
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()


# 1. PRZYGOTOWANIE DANYCH
df = pd.read_parquet('dataset.parquet')

C:\Users\kopcz\AppData\Local\Temp\ipykernel_56912\2032796478.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


FileNotFoundError: [Errno 2] No such file or directory: 'dataset.parquet'

In [ ]:
# Wyodrębniamy unikalne zadania (tasks)
all_tasks = df['task'].unique().tolist()
num_tasks = len(all_tasks)

# Przekształcamy dane: jeden wiersz = jedna cząsteczka
# Tworzymy macierz etykiet (Y) z NaN tam, gdzie nie ma danych
df_pivot = df.pivot_table(index='smiles', columns='task', values='label')

# Dołączamy deskryptory (X) i informację o splicie
feature_cols = [c for c in df.columns if c not in ['label', 'task', 'split', 'smiles', 'mask_1', 'mask_2', 'mask_3', 'mask_4']]
df_features_split = df.groupby('smiles').agg({**{c: 'first' for c in feature_cols}, 'split': 'first'})

# Łączymy w jeden finalny zestaw
final_df = df_features_split.join(df_pivot)

# Normalizacja deskryptorów (Kluczowe przy różnych skalach dipole/energy)
scaler = StandardScaler()
final_df[feature_cols] = scaler.fit_transform(final_df[feature_cols])

# Podział na zbiory zgodnie z Twoją kolumną 'split'
train_df = final_df[final_df['split'] == 'train']
test_df = final_df[final_df['split'] == 'test']


class ADMETDataset(Dataset):
    def __init__(self, dataframe, feature_list, task_list):
        self.X = torch.tensor(dataframe[feature_list].values, dtype=torch.float32)
        self.Y = torch.tensor(dataframe[task_list].values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


In [ ]:
class ADMETResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.4):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return x + self.block(x)


class ClassicalADMETModel(nn.Module):
    def __init__(self, input_dim=204, num_tasks=13):
        super().__init__()

        self.initial_projection = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        self.res_blocks = nn.Sequential(
            ADMETResidualBlock(256, dropout=0.4),
            ADMETResidualBlock(256, dropout=0.4)
        )

        self.post_shared = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU()
        )

        self.heads = nn.ModuleList([nn.Linear(128, 1) for _ in range(num_tasks)])

        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, x):
        x = self.initial_projection(x)
        x = self.res_blocks(x)
        shared_rep = self.post_shared(x)

        logits = torch.cat([head(shared_rep) for head in self.heads], dim=1)
        return logits


def uncertainty_loss(logits, targets, log_vars):
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    total_loss = 0
    raw_bce_sum = 0
    valid_tasks = 0

    mask = ~torch.isnan(targets)

    for i in range(logits.shape[1]):
        task_logit = logits[:, i][mask[:, i]]
        task_target = targets[:, i][mask[:, i]]

        if len(task_target) > 0:
            task_loss = loss_fn(task_logit, task_target).mean()
            raw_bce_sum += task_loss.item()
            valid_tasks += 1

            precision = torch.exp(-log_vars[i])
            total_loss += precision * task_loss + 0.5 * log_vars[i]

    avg_raw_bce = raw_bce_sum / valid_tasks if valid_tasks > 0 else 0
    return total_loss, avg_raw_bce


def generate_training_plot(train_losses, test_losses, filename='mtl_loss_plot.png'):
    plt.figure(figsize=(12, 6), dpi=100)

    # Rysowanie obu krzywych
    plt.plot(train_losses, label='Train Loss', color='royalblue', linewidth=2)
    plt.plot(test_losses, label='Test Loss', color='darkorange', linewidth=2)

    # Kosmetyka wykresu
    plt.title('Postęp Trenowania Modelu Multi-Task (ADMET)', fontsize=16, fontweight='bold')
    plt.xlabel('Epoka', fontsize=12)
    plt.ylabel('Wartość Funkcji Straty (Loss)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6) # Siatka ułatwia czytanie
    plt.legend(fontsize=12)

    plt.tight_layout()
    plt.savefig(filename)
    print(f"Wykres loss zapisany jako: {filename}")
    plt.close()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używam urządzenia: {device}")

batch_size = 256
train_loader = DataLoader(ADMETDataset(train_df, feature_cols, all_tasks),
                          batch_size=batch_size,
                          shuffle=True,
                          drop_last=True,
                          pin_memory=True)
val_loader = DataLoader(ADMETDataset(test_df, feature_cols, all_tasks), batch_size=batch_size)

model = ClassicalADMETModel(len(feature_cols), num_tasks).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)


# Przygotuj zmienne przed pętlą
best_val_loss = float('inf')
val_losses = []
train_losses = []

for epoch in range(100):
    # ========================================
    # 1. FAZA TRENINGU
    # ========================================
    model.train()
    running_train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/100 [Train]", unit="batch")

    for x, y in pbar:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)

        loss, raw_loss = uncertainty_loss(outputs, y, model.log_vars)
        loss.backward()
        optimizer.step()

        running_train_loss += raw_loss
        pbar.set_postfix(train_bce=raw_loss)

    epoch_train_loss = running_train_loss / len(train_loader)
    train_losses.append(epoch_train_loss)

    # ========================================
    # 2. FAZA WALIDACJI (EWALUACJI)
    # ========================================
    model.eval() # Wyłącza Dropout i BatchNorm
    running_val_loss = 0.0

    # torch.no_grad() to mus, wyłącza liczenie gradientów (oszczędza VRAM i czas)
    with torch.no_grad():
        for x_val, y_val in val_loader: # Musisz mieć osobny DataLoader!
            x_val, y_val = x_val.to(device), y_val.to(device)
            outputs_val = model(x_val)

            # W walidacji ignorujemy stratę z wagami (loss), interesuje nas tylko czyste BCE (raw_loss)
            _, raw_val_loss = uncertainty_loss(outputs_val, y_val, model.log_vars)
            running_val_loss += raw_val_loss

    epoch_val_loss = running_val_loss / len(val_loader)
    val_losses.append(epoch_val_loss)

    scheduler.step(epoch_val_loss)

    print(f"\nEpoch {epoch+1}: Train Loss = {epoch_train_loss:.4f} | Val Loss = {epoch_val_loss:.4f}")
    print(f"Bieżące log_vars: {model.log_vars.detach().cpu().numpy().round(3)}\n")

    # ZAPISUJEMY MODEL TYLKO GDY SPADA STRATA WALIDACYJNA
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "mtl_weights.pth")
        print(f" ---> Nowy rekord! Model zapisany (Val Loss: {best_val_loss:.4f})")

    if (epoch + 1) % 5 == 0:
        # Super byłoby zmodyfikować tę funkcję, żeby rysowała dwie linie: Train i Val
        generate_training_plot(train_losses, val_losses)

    # EARLY STOPPING OPARTY NA WALIDACJI
    if len(val_losses) > 10:
        recent_min = min(val_losses[-10:])
        if recent_min > best_val_loss:
            print(f"Early stopping w epoce {epoch+1}. Brak poprawy Val Loss od 10 epok.")
            break

In [ ]:
def evaluate_per_task(model, data_loader, task_names, device):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            # Sigmoid zamienia logity na prawdopodobieństwa (0-1)
            outputs = torch.sigmoid(model(x))

            all_preds.append(outputs.cpu().numpy())
            all_targets.append(y.numpy())

    # Łączymy batche w jedną macierz
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    print("\n=== WYNIKI EWALUACJI PER TASK ===")
    print(f"{'Zadanie':25} | {'AUROC':8} | {'Accuracy':8}")
    print("-" * 45)

    results = {}

    for i, task_name in enumerate(task_names):
        y_true = all_targets[:, i]
        y_pred_proba = all_preds[:, i]

        # Filtrowanie NaN
        mask = ~np.isnan(y_true)
        y_true_clean = np.round(y_true[mask]).astype(int)
        y_pred_proba_clean = y_pred_proba[mask]

        # Dla Accuracy potrzebujemy twardych klas (0 lub 1)
        # Przyjmujemy próg decyzyjny 0.5
        y_pred_class_clean = (y_pred_proba_clean >= 0.5).astype(int)

        if len(np.unique(y_true_clean)) > 1:
            # Liczymy obie metryki
            auroc = roc_auc_score(y_true_clean, y_pred_proba_clean)
            acc = accuracy_score(y_true_clean, y_pred_class_clean)

            results[task_name] = {'auroc': auroc, 'accuracy': acc}
            print(f"{task_name:25} | {auroc:.4f}   | {acc:.4f}")
        else:
            print(f"{task_name:25} | Brak wystarczających danych")

    return results

model.load_state_dict(torch.load("mtl_weights.pth"))
task_results = evaluate_per_task(model, val_loader, all_tasks, device)